In [1]:
from pathlib import Path
import sys
import logging

logger = logging.getLogger(__name__)
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New


In [2]:
import sys 
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

In [3]:
%load_ext autoreload
%autoreload 2 

In [4]:
import yaml
import pandas as pd
import numpy as np
import joblib

from src.features_notebook5 import create_all_features

from src.preprocessing import (
    prepare_xy,
    remove_unused_columns,
    get_feature_columns,
    build_preprocessor,
    fit_and_transform,
    get_feature_names,
    save_artifacts,
    save_processed_data
)

In [5]:
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

logger.info("Configuration loaded successfully.")

2026-09-21 16:36:21,476 - INFO - Configuration loaded successfully.


In [6]:
logger.info(config)

2026-09-21 16:36:22,998 - INFO - {'project': {'name': 'olist-mlops', 'random_state': 42}, 'database': {'user_env': 'DB_USER', 'password_env': 'DB_PASSWORD', 'host_env': 'DB_HOST', 'port_env': 'DB_PORT', 'name_env': 'DB_NAME'}, 'paths': {'ml_table': 'data/raw/olist_table.csv', 'labeled_table': 'data/processed/labeled_table.csv', 'train': 'data/splits/train.csv', 'validation': 'data/splits/validation.csv', 'test': 'data/splits/test.csv', 'features': 'models/features', 'eda_charts': 'data/eda_charts'}, 'data': {'target': 'is_late'}, 'features': {'categorical': ['customer_city', 'customer_state', 'main_payment_type'], 'id_columns': ['order_id', 'customer_id', 'customer_unique_id'], 'future_columns': ['order_delivered_carrier_date', 'order_delivered_customer_date'], 'raw_date_columns': ['order_purchase_timestamp']}, 'split': {'test_size': 0.3, 'validation_size': 0.5, 'random_state': 42}, 'logging': {'level': 'INFO', 'log_file': 'logs/app.log', 'format': '%(asctime)s | %(levelname)s | %(name

In [7]:
train_path = (
    PROJECT_ROOT
    / config["paths"]["train"]
)

val_path = (
    PROJECT_ROOT
    / config["paths"]["validation"]
)

test_path = (
    PROJECT_ROOT
    / config["paths"]["test"]
)

features_path = (
    PROJECT_ROOT
    / config["paths"]["features"]
)

logger.info("Train:%s", train_path)
logger.info("Validation:%s", val_path)
logger.info("Test:%s", test_path)
logger.info("Features:%s", features_path)

2026-09-21 16:36:32,090 - INFO - Train:c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New\data\splits\train.csv
2026-09-21 16:36:32,099 - INFO - Validation:c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New\data\splits\validation.csv
2026-09-21 16:36:32,099 - INFO - Test:c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New\data\splits\test.csv
2026-09-21 16:36:32,099 - INFO - Features:c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New\models\features


In [8]:
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

logger.info("Train:%s", train_df.shape)
logger.info("Validation:%s", val_df.shape)
logger.info("Test:%s", test_df.shape)

2026-09-21 16:37:00,995 - INFO - Train:(67533, 37)
2026-09-21 16:37:01,000 - INFO - Validation:(14471, 37)
2026-09-21 16:37:01,004 - INFO - Test:(14472, 37)


In [10]:
target = config["data"]["target"]

logger.info("Target:%s", target)

logger.info("\nTrain target distribution:")
logger.info(train_df[target].value_counts())

logger.info("\nValidation target distribution:")
logger.info(val_df[target].value_counts())

logger.info("\nTest target distribution:")
logger.info(test_df[target].value_counts())

2026-09-21 16:39:11,788 - INFO - Target:is_late


2026-09-21 16:39:11,791 - INFO - 
Train target distribution:
2026-09-21 16:39:19,753 - INFO - is_late
0    62054
1     5479
Name: count, dtype: int64
2026-09-21 16:39:20,023 - INFO - 
Validation target distribution:
2026-09-21 16:39:20,024 - INFO - is_late
0    13297
1     1174
Name: count, dtype: int64
2026-09-21 16:39:20,024 - INFO - 
Test target distribution:
2026-09-21 16:39:20,039 - INFO - is_late
0    13298
1     1174
Name: count, dtype: int64


In [11]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

logger.info(date_columns)

2026-09-21 16:39:22,202 - INFO - ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [12]:
train_df = create_all_features(
    train_df
)

val_df = create_all_features(
    val_df
)

test_df = create_all_features(
    test_df
)

In [13]:
logger.info("Train:%s", train_df.shape)
logger.info("Validation:%s", val_df.shape)
logger.info("Test:%s", test_df.shape)

2026-09-21 16:39:41,269 - INFO - Train:(67533, 41)
2026-09-21 16:39:41,275 - INFO - Validation:(14471, 41)
2026-09-21 16:39:41,281 - INFO - Test:(14472, 41)


In [14]:
new_features = [
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_weekday",
    "purchase_hour",
    "freight_to_price_ratio",
    "freight_per_item",
    "items_per_seller",
    "is_multi_seller",
    "is_weekend",
    "is_cross_state"
]

for feature in new_features:
    if feature in train_df.columns:
        logger.info(f"✓ {feature}")
    else:
        logger.error(f"- {feature} not available")

2026-09-21 16:39:43,239 - INFO - ✓ purchase_year
2026-09-21 16:39:43,244 - INFO - ✓ purchase_month
2026-09-21 16:39:43,248 - INFO - ✓ purchase_day
2026-09-21 16:39:43,250 - INFO - ✓ purchase_weekday
2026-09-21 16:39:43,250 - INFO - ✓ purchase_hour
2026-09-21 16:39:43,265 - INFO - ✓ freight_to_price_ratio
2026-09-21 16:39:43,272 - INFO - ✓ freight_per_item
2026-09-21 16:39:43,273 - INFO - ✓ items_per_seller
2026-09-21 16:39:43,273 - INFO - ✓ is_multi_seller
2026-09-21 16:39:43,273 - INFO - ✓ is_weekend
2026-09-21 16:39:43,288 - INFO - ✓ is_cross_state


In [15]:
y_train = train_df[target].copy()
y_val = val_df[target].copy()
y_test = test_df[target].copy()

X_train = train_df.drop(
    columns=[target]
)

X_val = val_df.drop(
    columns=[target]
)

X_test = test_df.drop(
    columns=[target]
)

In [16]:
id_columns = config["features"]["id_columns"]

future_columns = config["features"]["future_columns"]

raw_date_columns = config["features"]["raw_date_columns"]

In [17]:
X_train = remove_unused_columns(
    X_train,
    id_columns,
    future_columns,
    raw_date_columns
)

X_val = remove_unused_columns(
    X_val,
    id_columns,
    future_columns,
    raw_date_columns
)

X_test = remove_unused_columns(
    X_test,
    id_columns,
    future_columns,
    raw_date_columns
)

In [18]:
logger.info("X_train shape:%s", X_train.shape)
logger.info("X_val shape:%s", X_val.shape)
logger.info("X_test shape:%s", X_test.shape)

logger.info("\nRemaining columns:")
logger.info(X_train.columns.tolist())

2026-09-21 16:39:50,488 - INFO - X_train shape:(67533, 34)
2026-09-21 16:39:50,491 - INFO - X_val shape:(14471, 34)
2026-09-21 16:39:50,495 - INFO - X_test shape:(14472, 34)
2026-09-21 16:39:50,498 - INFO - 
Remaining columns:
2026-09-21 16:39:50,503 - INFO - ['order_status', 'order_approved_at', 'order_estimated_delivery_date', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'total_items', 'total_price', 'total_freight_value', 'avg_item_price', 'total_payment_value', 'max_installments', 'payment_count', 'main_payment_type', 'unique_products', 'unique_sellers', 'purchase_month', 'purchase_day', 'purchase_weekday', 'purchase_hour', 'is_weekend', 'multi_seller', 'freight_ratio', 'freight_per_item', 'total_weight', 'total_volume', 'avg_volume', 'max_volume', 'distance_km', 'is_cross_state', 'purchase_year', 'freight_to_price_ratio', 'items_per_seller', 'is_multi_seller']


In [19]:
categorical_features = config["features"]["categorical"]

categorical_features = [
    col
    for col in categorical_features
    if col in X_train.columns
]

logger.info("Categorical features:")
logger.info(categorical_features)

2026-09-21 16:39:52,680 - INFO - Categorical features:
2026-09-21 16:39:52,696 - INFO - ['customer_city', 'customer_state', 'main_payment_type']


In [20]:
numerical_features, categorical_features = (
    get_feature_columns(
        X_train,
        categorical_features
    )
)

logger.info("Numerical features:")
logger.info(numerical_features)

logger.info("\nCategorical features:")
logger.info(categorical_features)

2026-09-21 16:39:58,825 - INFO - Numerical features:
2026-09-21 16:39:58,840 - INFO - ['customer_zip_code_prefix', 'total_items', 'total_price', 'total_freight_value', 'avg_item_price', 'total_payment_value', 'max_installments', 'payment_count', 'unique_products', 'unique_sellers', 'multi_seller', 'freight_ratio', 'freight_per_item', 'total_weight', 'total_volume', 'avg_volume', 'max_volume', 'distance_km', 'is_cross_state', 'freight_to_price_ratio', 'items_per_seller']
2026-09-21 16:39:58,840 - INFO - 
Categorical features:
2026-09-21 16:39:58,849 - INFO - ['customer_city', 'customer_state', 'main_payment_type']


In [21]:
preprocessor = build_preprocessor(
    numerical_features,
    categorical_features
)

logger.info("Preprocessor created successfully.")

2026-09-21 16:40:00,072 - INFO - Preprocessor created successfully.


In [22]:
(
    X_train_processed,
    X_val_processed,
    X_test_processed
) = fit_and_transform(
    preprocessor,
    X_train,
    X_val,
    X_test
)

In [23]:
preprocessor.fit_transform(X_train)

<67533x3702 sparse matrix of type '<class 'numpy.float64'>'
	with 1620792 stored elements in Compressed Sparse Row format>

In [24]:
preprocessor.transform(X_val)

<14471x3702 sparse matrix of type '<class 'numpy.float64'>'
	with 347048 stored elements in Compressed Sparse Row format>

In [25]:
preprocessor.transform(X_test)

<14472x3702 sparse matrix of type '<class 'numpy.float64'>'
	with 347065 stored elements in Compressed Sparse Row format>

In [26]:
logger.info("X_train shape:%s", X_train.shape)
logger.info("X_val shape:%s", X_val.shape)
logger.info("X_test shape:%s", X_test.shape)

logger.info("\nTransformed:")
logger.info(
    "X_train_processed:%s",
    X_train_processed.shape
)

logger.info(
    "X_val_processed:%s",
    X_val_processed.shape
)

logger.info(
    "X_test_processed:%s",
    X_test_processed.shape
)

2026-09-21 16:40:20,450 - INFO - X_train shape:(67533, 34)
2026-09-21 16:40:20,453 - INFO - X_val shape:(14471, 34)
2026-09-21 16:40:20,457 - INFO - X_test shape:(14472, 34)
2026-09-21 16:40:20,460 - INFO - 
Transformed:
2026-09-21 16:40:20,465 - INFO - X_train_processed:(67533, 3702)
2026-09-21 16:40:20,473 - INFO - X_val_processed:(14471, 3702)
2026-09-21 16:40:20,473 - INFO - X_test_processed:(14472, 3702)


In [27]:
feature_names = get_feature_names(
    preprocessor
)

logger.info(
    "Number of final features:%s",
    len(feature_names)
)

logger.info(
    "Transformed columns:%s",
    X_train_processed.shape[1]
)

2026-09-21 16:40:24,097 - INFO - Number of final features:3702
2026-09-21 16:40:24,097 - INFO - Transformed columns:3702


In [28]:
assert len(feature_names) == X_train_processed.shape[1]

assert (
    X_train_processed.shape[1]
    == X_val_processed.shape[1]
)

assert (
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
)

logger.info("Feature dimensions are consistent.")

2026-09-21 16:40:26,367 - INFO - Feature dimensions are consistent.


In [29]:
save_artifacts(
    preprocessor=preprocessor,
    feature_names=feature_names,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    output_dir=features_path
)

logger.info("Preprocessing artifacts saved successfully.")

2026-09-21 16:40:31,272 - INFO - Preprocessing artifacts saved successfully.


In [30]:
save_processed_data(
    X_train_processed=X_train_processed,
    X_val_processed=X_val_processed,
    X_test_processed=X_test_processed,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    output_dir=features_path
)

logger.info("Processed data saved successfully.")

2026-09-21 16:40:39,130 - INFO - Processed data saved successfully.


In [35]:
logger.info("Artifacts:")

for file in sorted(features_path.iterdir()):
    logger.info(" -%s", file.name)

2026-09-21 16:42:43,137 - INFO - Artifacts:
2026-09-21 16:42:43,269 - INFO -  -feature_metadata.joblib
2026-09-21 16:42:43,271 - INFO -  -feature_names.joblib
2026-09-21 16:42:43,273 - INFO -  -preprocessor.joblib
2026-09-21 16:42:43,279 - INFO -  -X_test_processed.joblib
2026-09-21 16:42:43,281 - INFO -  -X_train_processed.joblib
2026-09-21 16:42:43,284 - INFO -  -X_val_processed.joblib
2026-09-21 16:42:43,287 - INFO -  -y_test.joblib
2026-09-21 16:42:43,289 - INFO -  -y_train.joblib
2026-09-21 16:42:43,300 - INFO -  -y_val.joblib


In [36]:
logger.info("Training target distribution:")
logger.info(y_train.value_counts())

logger.info("\nTraining target percentage:")
logger.info(
    y_train.value_counts(normalize=True) * 100
)

logger.info("\nValidation target distribution:")
logger.info(y_val.value_counts())

logger.info("\nValidation target percentage:")
logger.info(
    y_val.value_counts(normalize=True) * 100
)

2026-09-21 16:42:54,474 - INFO - Training target distribution:
2026-09-21 16:42:54,489 - INFO - is_late
0    62054
1     5479
Name: count, dtype: int64
2026-09-21 16:42:54,501 - INFO - 
Training target percentage:
2026-09-21 16:42:54,501 - INFO - is_late
0    91.886929
1     8.113071
Name: proportion, dtype: float64
2026-09-21 16:42:54,516 - INFO - 
Validation target distribution:
2026-09-21 16:42:54,531 - INFO - is_late
0    13297
1     1174
Name: count, dtype: int64
2026-09-21 16:42:54,533 - INFO - 
Validation target percentage:
2026-09-21 16:42:54,567 - INFO - is_late
0    91.887223
1     8.112777
Name: proportion, dtype: float64


In [38]:
logger.info("========== FINAL CHECKS ==========")

logger.info(
    "Train processed:%s",
    X_train_processed.shape
)

logger.info(
    "Validation processed:%s",
    X_val_processed.shape
)

logger.info(
    "Test processed:%s",
    X_test_processed.shape
)

logger.info(
    "Number of features:%s",
    len(feature_names)
)

logger.info(
    "Train target:%s",
    y_train.shape
)

logger.info(
    "Validation target:%s",
    y_val.shape
)

logger.info(
    "Test target:%s",
    y_test.shape
)

logger.info("\nPreprocessor:")
logger.info(type(preprocessor))

logger.info("\nAll checks completed successfully.")

2026-09-21 16:43:33,353 - INFO - ========== FINAL CHECKS ==========
2026-09-21 16:43:33,371 - INFO - Train processed:(67533, 3702)
2026-09-21 16:43:33,373 - INFO - Validation processed:(14471, 3702)
2026-09-21 16:43:33,373 - INFO - Test processed:(14472, 3702)
2026-09-21 16:43:33,388 - INFO - Number of features:3702
2026-09-21 16:43:33,388 - INFO - Train target:(67533,)
2026-09-21 16:43:33,388 - INFO - Validation target:(14471,)
2026-09-21 16:43:33,404 - INFO - Test target:(14472,)
2026-09-21 16:43:33,404 - INFO - 
Preprocessor:
2026-09-21 16:43:33,419 - INFO - <class 'sklearn.compose._column_transformer.ColumnTransformer'>
2026-09-21 16:43:33,419 - INFO - 
All checks completed successfully.


In [40]:
from src.preprocessing import load_preprocessor

saved_preprocessor = load_preprocessor(
    features_path / "preprocessor.joblib"
)

X_val_check = saved_preprocessor.transform(
    X_val
)

logger.info(
    "Original validation shape:%s",
    X_val_processed.shape
)

logger.info(
    "Reloaded validation shape:%s",
    X_val_check.shape
)

2026-09-21 16:43:53,194 - INFO - Original validation shape:(14471, 3702)
2026-09-21 16:43:53,208 - INFO - Reloaded validation shape:(14471, 3702)


In [41]:
assert (
    X_val_check.shape
   == X_val_processed.shape


)
logger.info(
"Saved preprocessor works correctly.")

2026-09-21 16:43:57,855 - INFO - Saved preprocessor works correctly.
